In [1]:
# 09_binding.ipynb
# Sensitivity analysis (Appendix D): when do the optimization structures bind?
# Varies saturation steepness, estimation-uncertainty scale, and budget tightness,
# reporting the submodular, robustness, and integrality prices for each.
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.optimize import milp, LinearConstraint, Bounds
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, TAB = BASE/"data", BASE/"results/tables"; TAB.mkdir(parents=True, exist_ok=True)
PVS=[f"PV{i}CIL" for i in range(1,6)]
def wpct(v,wt,p):
    m=~np.isnan(v); vs=np.sort(v[m]); ww=wt[m][np.argsort(v[m])]; return np.interp(p,np.cumsum(ww)/ww.sum(),vs)

stu,_=pyreadstat.read_sas7bdat(str(DATA/"bsgkori3.sas7bdat")); sch,_=pyreadstat.read_sas7bdat(str(DATA/"bcgkori3.sas7bdat"))
USED=["NISB","S_HOMLIT","S_INTNET","S_HISEI","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT","S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD","S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE","S_EXCOMP"]
for c in USED: stu.loc[(stu[c]>=995)|(stu[c]<=-995),c]=np.nan
stu.loc[(stu.NISB<-6)|(stu.NISB>6),"NISB"]=np.nan
LEV=["S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE"]; CTX=["NISB","S_HISEI","S_HOMLIT","S_INTNET","S_EXCOMP","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT","S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD"]
FE=LEV+CTX; d=stu[stu.TOTWGTS.notna()&stu.IDSCHOOL.notna()].copy(); W=d.TOTWGTS.to_numpy()
Xb=d[FE].to_numpy(float); Xc=Xb.copy()
for j,L in enumerate(LEV):
    t=wpct(Xb[:,j],W,0.75); col=Xc[:,j]; Xc[:,j]=np.where(np.isnan(col),col,np.maximum(col,t))
m=HistGradientBoostingRegressor(max_iter=400,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=0).fit(Xb,d.PV1CIL.to_numpy(float),sample_weight=W)
d["dCIL"]=m.predict(Xc)-m.predict(Xb)
g=d.groupby("IDSCHOOL").apply(lambda z:pd.Series(dict(mN=z.TOTWGTS.sum(),delta=np.average(z.dCIL,weights=z.TOTWGTS),
    nisb=np.average(z.NISB.fillna(z.NISB.mean()),weights=z.TOTWGTS),n=len(z),
    sd=np.sqrt(np.cov(z.dCIL,aweights=z.TOTWGTS)) if len(z)>1 else 0.0))).reset_index()
g["se"]=g.sd/np.sqrt(g.n); N=len(g); val=(g.mN*g.delta).to_numpy(); c1=g.mN.to_numpy()
shh=sch[["IDSCHOOL","C_HINRES"]].copy(); shh.loc[(shh.C_HINRES>=995)|(shh.C_HINRES<=-995),"C_HINRES"]=np.nan
g=g.merge(shh,on="IDSCHOOL",how="left"); hz=((g.C_HINRES-np.nanmean(g.C_HINRES))/np.nanstd(g.C_HINRES)).fillna(0)
c2=(g.mN*(1.0+0.5*np.clip(hz,-1.5,1.5))).to_numpy(); g["bucket"]=pd.qcut(g.nisb,3,labels=[0,1,2]).astype(int); bk=g.bucket.to_numpy()
def mkp(vals,caps,integer):
    r=milp(c=-vals,constraints=LinearConstraint(np.vstack([c1,c2]),-np.inf,list(caps)),integrality=(np.ones(N) if integer else np.zeros(N)),bounds=Bounds(0,1),options={"disp":False}); return -r.fun,r.x
caps=(0.33*c1.sum(),0.33*c2.sum())
# Stressor 1: saturation steepness
def submod_price(beta,caps):
    phi=lambda z: np.power(np.maximum(z,1e-9),beta); dphi=lambda z: beta*np.power(np.maximum(z,1e-9),beta-1)
    nb=3; nv=N+nb; rows=[np.concatenate([c1,np.zeros(nb)]),np.concatenate([c2,np.zeros(nb)])]; rhs=list(caps)
    for b in range(nb):
        Z=val[bk==b].sum()
        for zk in np.linspace(Z*0.02,Z,12):
            row=np.zeros(nv); row[np.where(bk==b)[0]]=-dphi(zk)*val[bk==b]; row[N+b]=1.0; rows.append(row); rhs.append(phi(zk)-dphi(zk)*zk)
    res=milp(c=np.concatenate([np.zeros(N),-np.ones(nb)]),constraints=LinearConstraint(np.array(rows),-np.inf,rhs),
        integrality=np.concatenate([np.ones(N),np.zeros(nb)]),bounds=Bounds(np.zeros(nv),np.concatenate([np.ones(N),[phi(val[bk==b].sum()) for b in range(nb)]])),options={"disp":False})
    xopt=res.x[:N].round().astype(bool); fval=lambda S: sum(phi(val[(bk==b)&S].sum()) for b in range(nb))
    _,ipx=mkp(val,caps,True); return 100*(fval(xopt)-fval(ipx.round().astype(bool)))/fval(ipx.round().astype(bool))
S1=pd.DataFrame([dict(beta=b,OPT_over_modular_pct=round(submod_price(b,caps),2)) for b in [0.9,0.7,0.5,0.3,0.2]])
S1.to_csv(TAB/"bind_saturation.csv",index=False); print("SATURATION:"); print(S1.to_string(index=False))
# Stressor 2: uncertainty scale
def robust_price(scale,caps,Gamma=30):
    dev=scale*(g.mN*g.se).to_numpy(); nom,_=mkp(val,caps,True); best=None
    for th in np.unique(np.append(dev,0.0)):
        adj=val-np.maximum(dev-th,0.0); _,xx=mkp(adj,caps,True); xx=xx.round(); obj=val@xx-np.maximum(dev-th,0.0)@xx-Gamma*th
        if best is None or obj>best[0]: best=(obj,xx)
    return 100*(nom-val@best[1])/nom
S2=pd.DataFrame([dict(uncertainty_scale=s,price_of_robustness_pct=round(robust_price(s,caps),2)) for s in [1,3,5,10]])
S2.to_csv(TAB/"bind_robust.csv",index=False); print("\nUNCERTAINTY:"); print(S2.to_string(index=False))
# Stressor 3: budget tightness
S3=pd.DataFrame([dict(budget_fraction=f,integrality_gap_pct=round(100*(mkp(val,(f*c1.sum(),f*c2.sum()),False)[0]-mkp(val,(f*c1.sum(),f*c2.sum()),True)[0])/mkp(val,(f*c1.sum(),f*c2.sum()),False)[0],3)) for f in [0.5,0.33,0.2,0.1,0.05]])
S3.to_csv(TAB/"bind_integrality.csv",index=False); print("\nBUDGET TIGHTNESS:"); print(S3.to_string(index=False))


SATURATION:
 beta  OPT_over_modular_pct
  0.9                  0.00
  0.7                  0.07
  0.5                  0.13
  0.3                  0.13
  0.2                  0.10


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


... (18 outputs left out)

HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();



UNCERTAINTY:
 uncertainty_scale  price_of_robustness_pct
                 1                     0.25
                 3                     4.12
                 5                     5.16
                10                   100.00


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();



BUDGET TIGHTNESS:
 budget_fraction  integrality_gap_pct
            0.50                0.110
            0.33                0.131
            0.20                0.141
            0.10                0.529
            0.05                0.294
